In [5]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

import os
import shutil

from tensorflow.keras.utils import image_dataset_from_directory

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow import keras

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, AveragePooling2D 

from tensorflow.keras.callbacks import EarlyStopping 
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

from tensorflow.keras.layers import BatchNormalization  
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from tensorflow.keras.callbacks import EarlyStopping

In [21]:
train_df = pd.read_csv("train.csv")

train_df.head()

,image_names,emergency_or_not
0,1503.jpg,0
1,1420.jpg,0
2,1764.jpg,0
3,1356.jpg,0
4,1117.jpg,0


In [ ]:
#create directories 
os.makedirs("dataset/emergency", exist_ok=True)
os.makedirs("dataset/non_emergency", exist_ok=True)

#Images source
source_folder = "train"

for index, row in train_df.iterrows():

    image_name = row['image_names']
    label = row['emergency_or_not']

#Create the full path to the image file.
    source_path = os.path.join(source_folder, image_name)

    # file paths for emergency and non-emergency
    if label == 0:
        destination_path = os.path.join("dataset/emergency", image_name)
    else:
        destination_path = os.path.join("dataset/non_emergency", image_name)

    # Copy image
    shutil.copy(source_path, destination_path)

print("Images organised successfully!")

In [8]:
#import the images
data_dir = "dataset"

train_ds = image_dataset_from_directory(data_dir,validation_split=0.1,subset="training",seed=42,batch_size=None)

test_ds = image_dataset_from_directory(data_dir,validation_split=0.1,subset="validation",seed=42,batch_size=None)

print(train_ds.class_names)

Found 1646 files belonging to 2 classes.
Using 1482 files for training.
Found 1646 files belonging to 2 classes.
Using 164 files for validation.
['emergency', 'non_emergency']


In [9]:
# Extract training data
x_train = []
y_train = []

for x, y in train_ds:
    x_train.append(np.uint8(x.numpy()))
    y_train.append(y.numpy())

x_train = np.array(x_train)
y_train = np.array(y_train)


# Extract testing data

x_test = []
y_test = []

for x, y in test_ds:
    x_test.append(np.uint8(x.numpy()))
    y_test.append(y.numpy())

x_test = np.array(x_test)
y_test = np.array(y_test)

In [ ]:
print(x_train.shape)
print(y_train.shape)

print(x_test.shape)
print(y_test.shape)

In [ ]:
#plot first image
plt.imshow(x_train[0], cmap="gray")
plt.title(f"Label: {y_train[0]}")
plt.axis("off")

In [ ]:
#to see the first five images
for i in range(5):
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
    plt.show()

In [13]:
#normalize
# divide the x_train and x_test arrays by 255.
x_train = x_train / 255
x_test = x_test / 255

width_npix = x_train.shape[1]
height_npix = x_train.shape[2]
print(width_npix )
print(height_npix)

256
256


### Data Argumentation

In [ ]:
train_datagen = ImageDataGenerator(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1, 
                                   horizontal_flip=True, vertical_flip=False, shear_range=0.10, zoom_range=0.10, validation_split=0.2)
train_datagen.fit(x_train) 

#create a randomly transformed image
new_image = train_datagen.random_transform(x_train[0]) 

#plot the image
plt.imshow(new_image, cmap = "gray")

### Baseline model 
The baseline CNN demonstrates solid overall performance, achieving an accuracy of 82.93%, with a notably high precision of 88.46% but a comparatively lower recall of 67.65%, indicating that while the model is highly reliable when predicting Emergency cases, it fails to identify a portion of true emergencies. This behaviour is reflected in the confusion matrix, where the model correctly classified 90 Emergency cases but missed 6, and produced 22 false alarms alongside 46 correct Non‑Emergency predictions, highlighting a precision‑oriented decision pattern. The loss curves further support this interpretation, showing a consistent downward trend in both training and validation loss across epochs with only minor fluctuations, suggesting effective learning and limited overfitting for a baseline architecture. Collectively, these results indicate that the model generalises reasonably well and is conservative in predicting emergencies, but would benefit from optimisation strategies aimed at improving recall to reduce missed Emergency cases, particularly in safety‑critical applications.

In [ ]:
model = Sequential()

#CONVOLUTION BLOCK 1(first hidden layer)
#add first convolutional layer
model.add(Conv2D(filters = 32, kernel_size = (3, 3),padding="same", input_shape = (width_npix, height_npix, 3), activation = 'relu')) 

#add pooling layer
model.add(MaxPooling2D(pool_size = (2, 2), padding="same")) 

#CONVOLUTION BLOCK 2
#second convolution layer 
model.add(Conv2D(filters = 64, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#CONVOLUTION BLOCK 3
#third convolution layer
model.add(Conv2D(filters = 128, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#add flatten layer(make it 1D)
model.add(Flatten())

#add layer
model.add(Dense(64, activation = 'relu'))

#add ouput layer
model.add(Dense(1, activation = 'sigmoid'))

model.summary()


model.compile(optimizer=Adam(),loss="binary_crossentropy",metrics=["accuracy"])

batch_size = 32

history = model.fit(train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="training"),epochs=20,
                    validation_data=train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="validation"))

#Plot the loss
history_df = pd.DataFrame(history.history) 
plt.plot(history_df["loss"], label = "Training") 
plt.plot(history_df["val_loss"], label = "Validation")
plt.title("Loss Curves – Baseline CNN")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


# Predict probabilities
y_prob = model.predict(x_test)


y_pred = (y_prob >= 0.5).astype(int)

#evaluate model performance
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
 
results_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall"],
    "Score": [accuracy, precision, recall]
})

print(results_df)

#plot the confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["Emergency", "Non-Emergency"],yticklabels=["Emergency", "Non-Emergency"])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

### Custom model 
1. Add regularization 

#### Add a drop out rate of 0.5
Why add regularization?
Regularisation was introduced because the baseline CNN exhibited early signs of overfitting and inconsistent generalisation, as reflected in both the performance metrics and the loss curves. Although the model achieved a relatively high accuracy of 82.93% and strong precision (88.46%), the noticeably lower recall (67.65%) indicated that the network was failing to generalise effectively to all Emergency cases, missing a meaningful proportion of true positives. This imbalance between precision and recall, combined with the confusion matrix showing 6 false negatives and 22 false positives, suggests that the model had learned patterns that were too specific to the training data rather than robust, generalisable features. The loss curves further supported this interpretation: while both training and validation loss decreased, the validation curve displayed mild fluctuations, implying that the model was beginning to rely on co-adapted features and was not learning stable representations. Introducing regularisation—starting with dropout at a rate of 0.5—helps mitigate these issues by preventing the network from over-relying on specific neurons, reducing co-adaptation, and encouraging the learning of more general, noise‑resistant features. This is expected to improve recall and overall generalisation, particularly important in a safety‑critical classification task such as emergency detection.

After introducing dropout regularisation at a rate of 0.5, the model’s performance metrics show a shift consistent with stronger regularisation pressure. Accuracy decreased slightly to 79.88% (from 82.93%), and precision remained relatively high at 87.23% (compared to 88.46%), indicating that the model still maintains good reliability when predicting Emergency cases. However, recall dropped more noticeably to 60.29% (from 67.65%), meaning the model is now missing more true Emergency cases than before. This behaviour is expected when strong dropout is introduced: the model becomes more conservative and less confident, which often reduces sensitivity (recall) in early regularisation stages. The updated confusion matrix supports this pattern, showing an increase in false negatives and false positives, reflecting the model’s reduced ability to capture subtle discriminative features after aggressively dropping neurons during training. The loss curves also illustrate this shift: training loss is higher and validation loss fluctuates more smoothly, indicating that the model is no longer overfitting but is now underfitting slightly due to the high dropout rate. Overall, these results confirm that regularisation is working as intended—reducing overfitting—but the dropout rate of 0.5 may be too strong for this dataset, suppressing the model’s capacity to learn sufficiently rich representations. A more moderate dropout rate (e.g., 0.2–0.3) or selective dropout placement may achieve a better balance between generalisation and recall.

In [ ]:
model = Sequential()

#CONVOLUTION BLOCK 1(first hidden layer)
#add first convolutional layer
model.add(Conv2D(filters = 32, kernel_size = (3, 3),padding="same", input_shape = (width_npix, height_npix, 3), activation = 'relu')) 

#add pooling layer
model.add(MaxPooling2D(pool_size = (2, 2), padding="same")) 

#CONVOLUTION BLOCK 2
#second convolution layer 
model.add(Conv2D(filters = 64, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#CONVOLUTION BLOCK 3
#third convolution layer
model.add(Conv2D(filters = 128, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#add flatten layer(make it 1D)
model.add(Flatten())

#add layer
model.add(Dense(64, activation = 'relu'))

#add drop out
model.add(Dropout(0.5))

#add ouput layer
model.add(Dense(1, activation = 'sigmoid'))

model.summary()

model.compile(optimizer=Adam(),loss="binary_crossentropy",metrics=["accuracy"])

batch_size = 32

history = model.fit(train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="training"),epochs=20,
                    validation_data=train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="validation"))

#Plot the loss
history_df = pd.DataFrame(history.history) 
plt.plot(history_df["loss"], label = "Training") 
plt.plot(history_df["val_loss"], label = "Validation")
plt.title("Loss Curves – Introduce drop out(0.5)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Predict probabilities
y_prob = model.predict(x_test)


y_pred = (y_prob >= 0.5).astype(int)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

#data frame for the results 
results_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall"],
    "Score": [accuracy, precision, recall]
})

print(results_df)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
    xticklabels=["Emergency", "Non-Emergency"],
    yticklabels=["Emergency", "Non-Emergency"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

#### Tune regularization to a drop out rate of 0.3
With a dropout rate of 0.3, the model achieves an accuracy of 80.49%, precision of 87.50%, and recall of 61.76%, representing a more balanced performance compared to the overly strong regularisation applied at 0.5. Precision remains high, indicating that the model continues to make reliable Emergency predictions, while recall improves relative to the 0.5‑dropout model (from 60.29% to 61.76%), suggesting that the network is now recovering some of its ability to detect true Emergency cases. This improvement is also reflected in the updated confusion matrix, where the number of false negatives remains low and false positives decrease slightly, indicating better discrimination between classes. The loss curves further support this trend: both training and validation loss decrease steadily with moderate fluctuations, showing that the model is no longer underfitting as severely as it was with dropout = 0.5, while still avoiding the overfitting observed in the baseline model. Overall, a dropout rate of 0.3 provides a more optimal regularisation strength for this dataset, improving generalisation without excessively suppressing the model’s learning capacity.

In [ ]:
model = Sequential()

#CONVOLUTION BLOCK 1(first hidden layer)
#add first convolutional layer
model.add(Conv2D(filters = 32, kernel_size = (3, 3),padding="same", input_shape = (width_npix, height_npix, 3), activation = 'relu')) 

#add pooling layer
model.add(MaxPooling2D(pool_size = (2, 2), padding="same")) 

#CONVOLUTION BLOCK 2
#second convolution layer 
model.add(Conv2D(filters = 64, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#CONVOLUTION BLOCK 3
#third convolution layer
model.add(Conv2D(filters = 128, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#add flatten layer(make it 1D)
model.add(Flatten())

#add layer
model.add(Dense(64, activation = 'relu'))

#add drop out
model.add(Dropout(0.3))

#add ouput layer
model.add(Dense(1, activation = 'sigmoid'))

model.summary()

model.compile(optimizer=Adam(),loss="binary_crossentropy",metrics=["accuracy"])

batch_size = 32

history = model.fit(train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="training"),epochs=20,
                    validation_data=train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="validation"))

#Plot the loss
history_df = pd.DataFrame(history.history) 
plt.plot(history_df["loss"], label = "Training") 
plt.plot(history_df["val_loss"], label = "Validation")
plt.title("Loss Curves – Introduce drop out(0.3)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Predict probabilities
y_prob = model.predict(x_test)


y_pred = (y_prob >= 0.5).astype(int)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

#data frame for the results 
results_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall"],
    "Score": [accuracy, precision, recall]
})

print(results_df)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
    xticklabels=["Emergency", "Non-Emergency"],
    yticklabels=["Emergency", "Non-Emergency"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

### Hyper parameter tuning 

#### Reduce learning rate of 0.0001
Reducing the learning rate to 0.0001 is justified because the baseline and regularised models exhibited unstable validation loss curves and a persistent gap between precision and recall, indicating that the optimizer was taking steps that were too large to converge smoothly to an optimal solution. The fluctuations in validation loss and the moderate recall values suggest that the model was overshooting good minima and failing to learn finer‑grained discriminative features, particularly for Emergency cases. Since dropout already increases training difficulty by reducing model capacity, a high learning rate further amplifies underfitting. Lowering the learning rate allows the network to update weights more gradually, improving stability, reducing noise in validation performance, and enabling the model to learn more subtle patterns necessary for better generalisation.

Reducing the learning rate to 0.0001 led to a clear decline in model performance, with accuracy dropping to 77.44%, precision to 86.05%, and recall to 54.41%, all lower than the results obtained using the default learning rate. The confusion matrix also showed an increase in both false positives and false negatives, indicating that the model struggled to learn discriminative features effectively at this slower update pace. The loss curves further support this conclusion: although both training and validation loss decreased, the convergence was noticeably slower and less effective, suggesting that the model was underfitting and failing to reach a good minimum within the available epochs. Because the default learning rate (0.001) allowed the model to learn more efficiently and produced stronger, more balanced metrics—especially higher recall—it is the more appropriate choice for this architecture and dataset. Therefore, the reduced learning rate does not improve generalisation and should not be used in place of the default.

In [ ]:
model = Sequential()

#CONVOLUTION BLOCK 1(first hidden layer)
#add first convolutional layer
model.add(Conv2D(filters = 32, kernel_size = (3, 3),padding="same", input_shape = (width_npix, height_npix, 3), activation = 'relu')) 

#add pooling layer
model.add(MaxPooling2D(pool_size = (2, 2), padding="same")) 

#CONVOLUTION BLOCK 2
#second convolution layer 
model.add(Conv2D(filters = 64, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#CONVOLUTION BLOCK 3
#third convolution layer
model.add(Conv2D(filters = 128, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#add flatten layer(make it 1D)
model.add(Flatten())

#add layer
model.add(Dense(64, activation = 'relu'))

#add drop out
model.add(Dropout(0.3))

#add ouput layer
model.add(Dense(1, activation = 'sigmoid'))

model.summary()

model.compile(optimizer=Adam(learning_rate=0.0001),loss="binary_crossentropy",metrics=["accuracy"])

batch_size = 32

history = model.fit(train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="training"),epochs=20,
                    validation_data=train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="validation"))


#Plot the loss
history_df = pd.DataFrame(history.history) 
plt.plot(history_df["loss"], label = "Training") 
plt.plot(history_df["val_loss"], label = "Validation")
plt.title("Loss Curves – Reduce learning rate(0.0001)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


# Predict probabilities
y_prob = model.predict(x_test)


y_pred = (y_prob >= 0.5).astype(int)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

#data frame for the results 
results_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall"],
    "Score": [accuracy, precision, recall]
})

print(results_df)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
    xticklabels=["Emergency", "Non-Emergency"],
    yticklabels=["Emergency", "Non-Emergency"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

#### Batch size 
Increase to 64
Increasing the batch size to 64 resulted in a substantial improvement in model performance, with accuracy rising to 90.24%, precision to 94.83%, and recall to 80.88%, marking the best balance between sensitivity and reliability achieved across all experiments. The confusion matrix shows a notable reduction in both false positives and false negatives, indicating that the model is now more effective at distinguishing Emergency from Non‑Emergency cases. This improvement can be attributed to the stabilising effect of a larger batch size, which provides smoother and more reliable gradient estimates during training. The loss curves further support this: both training and validation loss decrease steadily with reduced fluctuation, demonstrating more stable convergence compared to previous configurations. Unlike the smaller batch size models, which exhibited noisier validation behaviour and lower recall, the batch size of 64 allows the network to generalise better while maintaining high precision. Overall, these results show that increasing the batch size significantly enhances model performance, making batch size 64 the most effective configuration tested so far.


In [ ]:
model = Sequential()

#CONVOLUTION BLOCK 1(first hidden layer)
#add first convolutional layer
model.add(Conv2D(filters = 32, kernel_size = (3, 3),padding="same", input_shape = (width_npix, height_npix, 3), activation = 'relu')) 

#add pooling layer
model.add(MaxPooling2D(pool_size = (2, 2), padding="same")) 

#CONVOLUTION BLOCK 2
#second convolution layer 
model.add(Conv2D(filters = 64, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#CONVOLUTION BLOCK 3
#third convolution layer
model.add(Conv2D(filters = 128, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#add flatten layer(make it 1D)
model.add(Flatten())

#add layer
model.add(Dense(64, activation = 'relu'))

#add drop out
model.add(Dropout(0.3))

#add ouput layer
model.add(Dense(1, activation = 'sigmoid'))

model.summary()

model.compile(optimizer=Adam(),loss="binary_crossentropy",metrics=["accuracy"])

batch_size = 64

history = model.fit(train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="training"),epochs=20,
                    validation_data=train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="validation"))


#Plot the loss
history_df = pd.DataFrame(history.history) 
plt.plot(history_df["loss"], label = "Training") 
plt.plot(history_df["val_loss"], label = "Validation")
plt.title("Loss Curves – Increase batch size(64)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


# Predict probabilities
y_prob = model.predict(x_test)


y_pred = (y_prob >= 0.5).astype(int)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

#data frame for the results 
results_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall"],
    "Score": [accuracy, precision, recall]
})

print(results_df)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
    xticklabels=["Emergency", "Non-Emergency"],
    yticklabels=["Emergency", "Non-Emergency"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

#### Introduce early stopping

Introducing early stopping resulted in a noticeable decline in model performance, with accuracy decreasing to 79.88%, precision to 79.66%, and recall to 69.12%, all of which are lower than the metrics achieved by the optimised model using a batch size of 64. While early stopping is designed to prevent overfitting by halting training once validation performance stops improving, in this case it caused the model to stop learning prematurely. The reduced precision indicates that the model became less reliable in identifying Emergency cases correctly, while the drop in recall shows that it also missed more true Emergency instances compared to the best-performing configuration. This suggests that the model still required additional epochs to fully converge, and early stopping interrupted this process before the network could stabilise and refine its feature representations. Given that the batch size of 64 produced significantly higher accuracy, precision, and recall without signs of overfitting, early stopping is not beneficial for this particular architecture and dataset. Therefore, the results justify not using early stopping and instead retaining the configuration that allows the model to train for the full number of epochs.

In [ ]:
model = Sequential()

#CONVOLUTION BLOCK 1(first hidden layer)
#add first convolutional layer
model.add(Conv2D(filters = 32, kernel_size = (3, 3),padding="same", input_shape = (width_npix, height_npix, 3), activation = 'relu')) 

#add pooling layer
model.add(MaxPooling2D(pool_size = (2, 2), padding="same")) 

#CONVOLUTION BLOCK 2
#second convolution layer 
model.add(Conv2D(filters = 64, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#CONVOLUTION BLOCK 3
#third convolution layer
model.add(Conv2D(filters = 128, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#add flatten layer(make it 1D)
model.add(Flatten())

#add layer
model.add(Dense(64, activation = 'relu'))

#add drop out
model.add(Dropout(0.3))

#add ouput layer
model.add(Dense(1, activation = 'sigmoid'))

model.summary()

model.compile(optimizer=Adam(),loss="binary_crossentropy",metrics=["accuracy"])

batch_size = 64


early_stop = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)


history = model.fit(train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="training"),epochs=20,
                    validation_data=train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="validation"),callbacks=[early_stop])


#Plot the loss
history_df = pd.DataFrame(history.history) 
plt.plot(history_df["loss"], label = "Training") 
plt.plot(history_df["val_loss"], label = "Validation")
plt.title("Loss Curves – Increase batch size(64)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


# Predict probabilities
y_prob = model.predict(x_test)


y_pred = (y_prob >= 0.5).astype(int)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

#data frame for the results 
results_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall"],
    "Score": [accuracy, precision, recall]
})

print(results_df)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))

sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
    xticklabels=["Emergency", "Non-Emergency"],
    yticklabels=["Emergency", "Non-Emergency"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

Through a structured and iterative tuning process, several architectural and training adjustments were explored to improve the performance of the emergency vehicle classification model. Regularisation strategies, learning rate modifications, and early stopping were each evaluated, but none produced improvements comparable to the gains achieved by increasing the batch size. The model trained with a batch size of 64, combined with dropout at 0.3 and the default Adam learning rate, consistently delivered the strongest results, achieving 90.24% accuracy, 94.83% precision, and 80.88% recall. These metrics, supported by stable loss curves and a well‑balanced confusion matrix, demonstrate superior generalisation and reliable class separation. Based on this evidence, the batch‑size‑64 configuration is selected as the final model, offering the most effective and robust solution for the emergency versus non‑emergency image classification task.

In [ ]:
best_model = Sequential()

#CONVOLUTION BLOCK 1(first hidden layer)
#add first convolutional layer
best_model.add(Conv2D(filters = 32, kernel_size = (3, 3),padding="same", input_shape = (width_npix, height_npix, 3), activation = 'relu')) 

#add pooling layer
best_model.add(MaxPooling2D(pool_size = (2, 2), padding="same")) 

#CONVOLUTION BLOCK 2
#second convolution layer 
best_model.add(Conv2D(filters = 64, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
best_model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#CONVOLUTION BLOCK 3
#third convolution layer
best_model.add(Conv2D(filters = 128, kernel_size = (3, 3),padding="same", activation = 'relu')) 

#pooling layer 
best_model.add(MaxPooling2D(pool_size = (2, 2), padding="same"))

#add flatten layer(make it 1D)
best_model.add(Flatten())

#add layer
best_model.add(Dense(64, activation = 'relu'))

#add drop out
best_model.add(Dropout(0.3))

#add ouput layer
best_model.add(Dense(1, activation = 'sigmoid'))

best_model.summary()

best_model.compile(optimizer=Adam(),loss="binary_crossentropy",metrics=["accuracy"])

batch_size = 64

history = best_model.fit(train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="training"),epochs=20,
                    validation_data=train_datagen.flow(x_train,y_train,batch_size=batch_size,subset="validation"))



In [ ]:
#save the model

os.makedirs("model", exist_ok=True)

best_model.save("model/emergency_vehicle_classifier.keras")

In [ ]:
#verify if you can load the model
from tensorflow.keras.models import load_model

loaded_model = load_model("../model/emergency_vehicle_classifier.keras")

loaded_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 131072)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     8,388,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,445,957 (97.07 MB)

 Trainable params: 8,481,985 (32.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 16,963,972 (64.71 MB)

In [11]:
prediction = loaded_model.predict(x_test[:1])

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step
[[0.9999842]]


In [25]:
#Create a prediction function 
from tensorflow.keras.preprocessing import image
import numpy as np

def predict_vehicle(img_path):

    img = image.load_img(img_path, target_size=(256, 256))

    img_array = image.img_to_array(img)

    img_array = img_array / 255.0

    img_array = np.expand_dims(img_array, axis=0)

    probability = loaded_model.predict(img_array, verbose=0)[0][0]

    if probability >= 0.5:
        label = "Emergency Vehicle"
    else:
        label = "Non-Emergency Vehicle"

    return label, probability

In [ ]:
#Test the prediction function(with a known emergency vehicle)
predict_vehicle("C:/Users/ADMIN/OneDrive - hull.ac.uk/Documents/Emergency_Vehicles_Classifier/test/13.jpg")

('Emergency Vehicle', 0.9993495)

In [ ]:
#Test the prediction function(with a known non emergency vehicle)
predict_vehicle("C:/Users/ADMIN/OneDrive - hull.ac.uk/Documents/Emergency_Vehicles_Classifier/test/184.jpg")

('Non-Emergency Vehicle', 0.20648684)